In [1]:
import os
import json
from collections import Counter

INPUT_DATA_PATH = '/kaggle/input/notebooks/dadhichisarker/qualcom-ai-preprocessing'
LIST_PATH = '/kaggle/input/notebooks/dadhichisarker/qualcom-ai-preprocessing/processed_files_list.json'
INDEX_PATH = '/kaggle/input/notebooks/dadhichisarker/qualcom-ai-preprocessing/class_indices.json'

if os.path.exists(LIST_PATH) and os.path.exists(INDEX_PATH):
    with open(LIST_PATH, 'r') as f:
        data_list = json.load(f)
    with open(INDEX_PATH, 'r') as f:
        class_to_idx = json.load(f)

    num_classes = len(class_to_idx)
    label_counts = Counter(item['label'] for item in data_list)

    print(f"✅ Total processed videos found: {len(data_list)}")
    print(f"✅ Total classes found: {num_classes}")
    print(f"📂 Sample path: {data_list[0]['path']}")
    print(f"📉 Min samples in a class: {min(label_counts.values())}")
    print(f"📈 Max samples in a class: {max(label_counts.values())}")
else:
    raise FileNotFoundError("❌ processed_files_list.json or class_indices.json not found. Check Kaggle input dataset path.")

✅ Total processed videos found: 5860
✅ Total classes found: 91
📂 Sample path: air_jump_rope/00065305.pth
📉 Min samples in a class: 1
📈 Max samples in a class: 70


In [2]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

class QualcommExerciseModel(nn.Module):
    def __init__(self, num_classes=91):
        super(QualcommExerciseModel, self).__init__()
        # MobileNetV3-Small features
        # Note: Dragonwing/QNN performs best with Hardswish/Hardsigmoid which are standard in MBV3
        backbone_model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        self.backbone = backbone_model.features
        
        # We replace the global pooling with a 3D pool to handle (C, T, H, W)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        
        # Classifier head
        # MBV3 small features output 576 channels
        self.classifier = nn.Sequential(
            nn.Linear(576, 256),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # Input shape: (Batch, Channels, Frames, H, W) -> (B, 3, 16, 112, 112)
        b, c, f, h, w = x.shape
        
        # 1. Reshape to treat frames as a batch for the 2D backbone
        # (B, C, F, H, W) -> (B, F, C, H, W) -> (B*F, C, H, W)
        x = x.permute(0, 2, 1, 3, 4).reshape(-1, c, h, w)
        
        # 2. Extract 2D features
        x = self.backbone(x)  # Output: (B*F, 576, 4, 4)
        
        # 3. Reshape back to 3D for temporal pooling
        # (B*F, 576, H', W') -> (B, F, 576, H', W') -> (B, 576, F, H', W')
        _, c_out, h_out, w_out = x.shape
        x = x.reshape(b, f, c_out, h_out, w_out).permute(0, 2, 1, 3, 4)
        
        # 4. Global Spatiotemporal Pooling
        x = self.avgpool(x)  # (B, 576, 1, 1, 1)
        x = torch.flatten(x, 1)
        
        return self.classifier(x)

# Initialize on device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QualcommExerciseModel(num_classes=91).to(device)
print(f"🚀 Optimized Architecture initialized on {device}")


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 99.3MB/s]


🚀 Optimized Architecture initialized on cuda


In [3]:
import torch
from torch.utils.data import Dataset
import os

class QEVDTensorDataset(Dataset):
    def __init__(self, data_list, root_dir):
        self.data_list = data_list
        self.root_dir = root_dir

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        file_path = os.path.join(self.root_dir, item['path'])
        label = item['label']
        
        # টেনসর লোড করা
        # Shape: (16, 3, 112, 112)
        # Note: Preprocessing already applied normalization and converted to float tensor.
        video_tensor = torch.load(file_path).float() 
        
        # (Frames, Channels, H, W) -> (Channels, Frames, H, W) 
        # আমাদের 3D মডেলের জন্য এই ফরম্যাট লাগবে
        video_tensor = video_tensor.permute(1, 0, 2, 3)
        
        return video_tensor, label

In [4]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
from collections import Counter
import torch

# 1. Handle extreme class imbalance safely during train-test split
labels = [item['label'] for item in data_list]
label_counts = Counter(labels)

# Group items into common (>= 2 vids) and rare (< 2 vids)
common_items = [item for item in data_list if label_counts[item['label']] >= 2]
rare_items = [item for item in data_list if label_counts[item['label']] < 2]
common_labels = [item['label'] for item in common_items]

# Stratified split only on classes with enough samples
train_common, val_list = train_test_split(
    common_items,
    test_size=0.2,
    random_state=42,
    stratify=common_labels
)

# Manually append the 1-sample classes directly into training so sklearn doesn't crash
train_list = train_common + rare_items

# 2. Dataset objects
ROOT_DIR = '/kaggle/input/notebooks/dadhichisarker/qualcom-ai-preprocessing/processed_tensors'
train_dataset = QEVDTensorDataset(train_list, ROOT_DIR)
val_dataset = QEVDTensorDataset(val_list, ROOT_DIR)

# 3. Weighted sampler for class imbalance handling
train_labels = [item['label'] for item in train_list]
class_sample_counts = np.bincount(train_labels, minlength=91)
class_weights = 1.0 / np.clip(class_sample_counts, a_min=1, a_max=None)
sample_weights = [class_weights[label] for label in train_labels]

weighted_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# 4. DataLoaders
# Need pin_memory=True for fast GPU ops
train_loader = DataLoader(train_dataset, batch_size=16, sampler=weighted_sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"📊 Training Samples: {len(train_dataset)} (Including {len(rare_items)} rare 1-sample videos)")
print(f"📊 Validation Samples: {len(val_dataset)}")
print(f"📊 Train batches/epoch: {len(train_loader)}")

📊 Training Samples: 4688 (Including 1 rare 1-sample videos)
📊 Validation Samples: 1172
📊 Train batches/epoch: 293


In [5]:
import torch.optim as optim
import numpy as np

# Build class-weighted loss from the training split
train_labels = [item['label'] for item in train_list]
class_sample_counts = np.bincount(train_labels, minlength=91)
class_weights = 1.0 / np.clip(class_sample_counts, a_min=1, a_max=None)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Slower decay than StepLR helps this dataset
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)
print("✅ Loss/optimizer/scheduler configured for imbalanced multiclass training")

✅ Loss/optimizer/scheduler configured for imbalanced multiclass training


In [6]:
import torch

# ১. সরাসরি চেক করা
if torch.cuda.is_available():
    print(f"✅ কুপেল্লা মামা! GPU পাওয়া গেছে।")
    print(f"🔥 ডিভাইসের নাম: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("❌ কপাল খারাপ! GPU পাওয়া যায়নি, CPU-তে রান হবে।")
    device = torch.device("cpu")

# ২. তোমার মডেলের ডিভাইস চেক করা (যদি আগে মডেল ডিফাইন করে থাকো)
print(f"🚀 বর্তমানে মডেলটি আছে: {next(model.parameters()).device}")

✅ কুপেল্লা মামা! GPU পাওয়া গেছে।
🔥 ডিভাইসের নাম: Tesla T4
🚀 বর্তমানে মডেলটি আছে: cuda:0


In [7]:
import torch

# ডিভাইস ডিটেক্ট করা
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == 'cuda':
    print(f"✅ কুপেল্লা মামা! {torch.cuda.get_device_name(0)} জিপিইউ রেডি।")
else:
    print("⚠️ সাবধান! জিপিইউ পাওয়া যায়নি, সিপিইউ-তে রান হবে। (Kaggle Settings চেক করো)")

# IMPORTANT: Do NOT recreate model here, otherwise optimizer will point to old parameters.
model = model.to(device)

# চেক করে দেখা মডেল কোথায় আছে
print(f"🚀 মডেল এখন আছে: {next(model.parameters()).device}")

✅ কুপেল্লা মামা! Tesla T4 জিপিইউ রেডি।
🚀 মডেল এখন আছে: cuda:0


In [8]:
from tqdm import tqdm
import torch

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(loader, desc='Training')
    for inputs, labels in loop:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        loop.set_postfix(loss=running_loss / max(1, len(loader)), acc=100.0 * correct / max(1, total))

    return running_loss / len(loader), 100.0 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100.0 * correct / total

/tmp/ipykernel_55/1070194454.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))


In [9]:
# --- Training loop with best-checkpoint saving ---
EPOCHS = 20
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS}:")
    print(f"🔥 Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    print(f"📉 Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"📌 LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("-" * 30)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_qevd_model.pth')
        print(f"✅ New best model saved with Val Acc: {best_val_acc:.2f}%")

print(f"🏁 Training complete. Best Val Acc: {best_val_acc:.2f}%")

Training:   0%|          | 0/293 [00:00<?, ?it/s]/tmp/ipykernel_55/1070194454.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
Training: 100%|██████████| 293/293 [01:03<00:00,  4.59it/s, acc=6.34, loss=2.99] 


Epoch 1/20:
🔥 Train Acc: 6.34% | Val Acc: 0.68%
📉 Train Loss: 2.9876 | Val Loss: 5.0669
📌 LR: 0.000298
------------------------------
✅ New best model saved with Val Acc: 0.68%


Training: 100%|██████████| 293/293 [00:30<00:00,  9.60it/s, acc=10.9, loss=2.56] 


Epoch 2/20:
🔥 Train Acc: 10.86% | Val Acc: 6.83%
📉 Train Loss: 2.5647 | Val Loss: 4.6751
📌 LR: 0.000293
------------------------------
✅ New best model saved with Val Acc: 6.83%


Training: 100%|██████████| 293/293 [00:23<00:00, 12.43it/s, acc=20.8, loss=2.49] 


...

Epoch 18/20:
🔥 Train Acc: 90.83% | Val Acc: 38.57%
📉 Train Loss: 1.1718 | Val Loss: 3.8437
📌 LR: 0.000008
------------------------------


Training: 100%|██████████| 293/293 [00:17<00:00, 16.85it/s, acc=91.6, loss=1.15] 


Epoch 19/20:
🔥 Train Acc: 91.62% | Val Acc: 38.31%
📉 Train Loss: 1.1519 | Val Loss: 3.8512
📌 LR: 0.000003
------------------------------


Training: 100%|██████████| 293/293 [00:17<00:00, 16.78it/s, acc=91.5, loss=1.12] 


Epoch 20/20:
🔥 Train Acc: 91.53% | Val Acc: 38.74%
📉 Train Loss: 1.1198 | Val Loss: 3.8484
📌 LR: 0.000001
------------------------------
🏁 Training complete. Best Val Acc: 39.08%


In [ ]:
# --- 1. Export to ONNX ---
import torch.onnx

def export_model_to_onnx(model, output_path="qualcomm_exercise_model.onnx"):
    model.eval()
    # Input shape: (Batch, Channels, Frames, H, W) -> (1, 3, 16, 112, 112)
    dummy_input = torch.randn(1, 3, 16, 112, 112).to(device)
    
    torch.onnx.export(
        model, 
        dummy_input, 
        output_path, 
        export_params=True, 
        opset_version=14,  # QNN works well with 13-14 usually
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        # Dynamic axes are not supported on Dragonwing/QNN, so we stick to fixed shapes
        # as the competition specifies 16 frames and 112x112 resize
    )
    print(f"✅ Model exported to: {output_path}")

# Export after training is completed
# export_model_to_onnx(model)

# --- 2. AI Hub Compilation & Profiling ---
# Note: You need your Qualcomm AI Hub API Token for this.
# pip install qai_hub

def compile_and_profile_on_ai_hub(onnx_path="qualcomm_exercise_model.onnx"):
    import qai_hub as hub
    
    # 1. Device selection
    # Competition specifies Qualcomm Dragonwing IQ-9075 EVK
    device_name = "Qualcomm Dragonwing IQ-9075 EVK"
    device = hub.get_devices(device_name)[0]
    
    # 2. Upload model
    hub_model = hub.upload_model(onnx_path)
    
    # 3. Submit Compile Job
    # Targeting Dragonwing 
    print(f"🛠️ Starting compilation for {device_name}...")
    compile_job = hub.submit_compile_job(
        model=hub_model,
        device=device,
        input_specs=dict(input=(1, 3, 16, 112, 112)),
    )
    
    # Wait for completion
    compile_result = compile_job.get_target_model()
    
    # 4. Profile Latency
    print(f"⏱️ Starting profiling...")
    profile_job = hub.submit_profile_job(
        model=compile_result,
        device=device,
    )
    
    # Wait for completion
    profile_data = profile_job.download_profile()
    latency = profile_data['execution_summary']['estimated_inference_time'] / 1000.0 # Convert to ms
    
    print(f"✅ Profiling Complete!")
    print(f"📉 Estimated Latency: {latency:.2f} ms")
    
    if latency < 34.0:
        print("🚀 PASSED! Model is within the 34ms budget.")
    else:
        print("⚠️ FAILED! Model exceeds the 34ms budget. Consider MBV3-Small or pruning.")
    
    # IMPORTANT: Sharing for LPCVC submission
    # compile_job.modify_sharing(add_emails=['lowpowervision@gmail.com'])
    # print("✅ Shared compile job with LPCVC organizers.")
    
    return compile_job.id

# run after export
# compile_and_profile_on_ai_hub()
